# Delta vs. Parquet: Curiosity & Challenge Questions

---

### 1️⃣ Vanilla Parquet Challenge

**Q:** If you overwrite data in a Parquet table (S3 path) and a query engine (like Athena) is scanning the directory at the same time, what happens if your Spark job crashes midway?

**Why it matters:**

* What visibility and data integrity guarantees do you have? Could readers see partial, corrupted, or missing data?

**A:**

* There’s no atomicity—readers can see a partially written dataset, missing files, or even lose data if old files are deleted before the new ones finish writing. No rollback is possible; you’re left to manually repair the table.

---

### 2️⃣ Delta Lake: Atomic Writes & ACID

**Q:** How does Delta ensure that readers never see half-written or in-progress data—even during concurrent writes or job failures?

**Why it matters:**

* How can a lakehouse table offer ACID properties atop “dumb” object storage like S3?

**A:**

* Delta writes new Parquet files first, then only “commits” the changes by appending a transaction log in \_delta\_log/. Readers always reference the latest successful commit—so they either see the old state or the new, never a partial or broken table.

---

### 3️⃣ Rollback & Time Travel

**Q:** If you accidentally delete or corrupt data, can you roll back in Parquet? How does Delta allow you to “undo” changes or audit history?

**Why it matters:**

* What operational safety nets exist for real data lakes?

**A:**

* Parquet has no versioning—once data is overwritten or deleted, it’s gone. Delta records every change in commit logs, letting you read data “as of” any previous version (time travel), perform audits, or restore after bad writes.

---

### 4️⃣ Schema Evolution & Enforcement

**Q:** What happens if a new writer tries to add a column or change a schema in Parquet vs Delta?

**Why it matters:**

* How do you prevent schema drift and silent failures in a data lake?

**A:**

* Parquet will just write new files—even with a different schema—silently causing downstream confusion. Delta enforces the schema by default, throwing errors if you violate it, and supports controlled evolution (add/drop columns) with full tracking.

---

### 5️⃣ Upserts & MERGE INTO

**Q:** How would you perform an “upsert” (update if exists, insert if not) or a CDC merge on Parquet? How is it different in Delta?

**Why it matters:**

* What’s the cost and risk of making data warehouse operations “lake-native”?

**A:**

* With Parquet, you must manually read, join, filter, and rewrite entire files—risky, slow, and error-prone. Delta exposes a simple MERGE INTO API, which handles all logic atomically, logs every change, and enables rollbacks/time travel.

---

### 6️⃣ Vacuum & Cleanup

**Q:** As you build up versions, how do you manage stale data or “orphaned” files? What’s the difference in cleanup for Parquet vs Delta?

**Why it matters:**

* What prevents a lakehouse from ballooning in storage or reading dead files?

**A:**

* Parquet leaves all file management to you—no automated cleanup. Delta offers VACUUM to remove files that are no longer referenced by any table version, after a configurable retention period.

---

### 7️⃣ Operational Safety & Auditability

**Q:** If a failure occurs during append or overwrite, or multiple jobs write at once, how do you guarantee auditability and data integrity?

**Why it matters:**

* What guarantees can you offer to downstream consumers?

**A:**

* Delta’s transactional log and isolation model ensure only one writer wins per transaction, and all operations are audit-trailed with who/when/what metadata. Parquet provides no such history or safety.

---

## 🔑 How to Use This List

* **Before interviews:** Challenge yourself with these Qs—can you explain both sides?
* **For documentation:** Use them as FAQ headers for internal wikis.
* **In team meetings:** Pose them to test system design choices and raise awareness of hidden risks.

**Pro tip:**
If you ever feel a concept is foggy, ask:

> “How would I guarantee this in raw Parquet? How does Delta (or Iceberg/Hudi) make it safe and easy?”

---

# 🚨 Misconceptions & Course Corrections

---

### ❌ Misconception:

“If I don’t use MERGE INTO, Delta is basically the same as vanilla Parquet except for time travel.”

**Course Correction:**
Even without using MERGE, every write, append, update, or delete in Delta is tracked in the transaction log—giving you not just time travel, but:

* Atomic commits
* Schema enforcement
* Failure recovery
* VACUUM for cleaning old files
* Consistent snapshot isolation for all readers

So, Delta is Parquet++ for all write patterns—not just when you use advanced commands.

---

### ❌ Misconception:

“Time travel and vacuum only matter when I do updates or deletes.”

**Course Correction:**
You get versioning and time travel on every operation—append, overwrite, schema change, etc.
VACUUM is about cleaning up unreferenced files from any older version, not just after updates or deletes.

---

### ❌ Misconception:

“Delta Lake achieves atomicity by overwriting data in place and then updating a pointer.”

**Course Correction:**
Delta never deletes old data first. It always:

* Writes new files
* Commits a new log entry
* Makes new data visible only after commit succeeds

No reader ever sees an in-between state. The pointer (log) is the single source of truth.

---

### ❌ Misconception:

“Vanilla Parquet overwrites are safe because the write command completes the whole operation.”

**Course Correction:**
Parquet writes are not transactional—if your job crashes or the network fails, you may lose data, end up with partial files, or corrupt the table. No automatic rollback.

---

### ❌ Misconception:

“Schema evolution is just as easy in Parquet as in Delta.”

**Course Correction:**
Delta tracks schema changes per version and enforces them on writes. Vanilla Parquet lets you write mismatched schemas—often breaking downstream jobs silently.

---

## 🌱 How to Use These

* Whenever you “think it’s just like Parquet,” revisit these and look for the invisible protections Delta provides.
* Challenge your teammates or yourself with: “What would break if this was just Parquet?”
